# Python 离群点检测（Outlier Detection）系列教程 · 第0章 导论

> 本系列定位：面向具备 Python 和基础机器学习基础的读者；
> 主题范围：**聚焦无监督的离群点检测（Outlier Detection 严格定义）**——给一堆没有标签的数据，找出偏离主体的少数样本。监督式与单类方法不是本系列主线。
> 写作主线：**先用"算法家族全景图"建立地图，再一章一个专用包、按工业重要程度依次落地**。
> 本章为系列开篇：界定概念 → 给出算法家族地图 → 给出选型决策框架 → 梳理生态与评估范式 → 预告完整章节路线。

---

## 本章你将获得

1. 说清离群点检测（Outlier Detection）到底是怎么回事——无监督、无标签、找偏离主体的样本，并认识它的三种目标形态；
2. 拿到一张**算法家族全景图**，知道后续各章的算法分别属于哪个家族；
3. 拿到一个**三步选型决策框架**，面对新问题知道从哪类算法起步；
4. 理解离群点检测"排序优先"的评估范式，以及无标签时如何构建验证集、如何定阈值；
5. 跑通一个十行以内的端到端小例子，验证环境可用。

---

## 1. 什么是离群点检测（Outlier Detection）

### 1.1 一句话定义

> **离群点检测，就是在一堆没有任何标签的数据里，找出那些"跟大家不一样"的少数样本。**

- **输入**：一批样本，**没有任何"这是正常 / 这是异常"的标注**；
- **输出**：给每个样本打一个分数（anomaly score），**分数越高越可疑**。

举个直观的例子：某账户历史交易金额都在几十到几百元之间，突然出现一笔 50000 元的交易。没有人事先给这笔交易贴上"欺诈"标签，算法只是发现它离其他交易太远，于是给它一个很高的异常分数。整个过程不需要任何标签——这就是无监督学习。

这正是 Hawkins 的经典定义：

> "一个异常是一个观测，它和其他观测差异巨大，以至于让人怀疑它是由另一种机制生成的。"

**算法凭什么判断"不一样"？** 它靠的是数据本身的结构，而不是标准答案：某个样本周围有多空旷（**密度**）、它离邻居有多远（**距离**）、它有多容易被随机切分隔离出来（**隔离程度**）。后面所有章节的算法，底层逻辑都逃不出这几类。

> 💡 关键认知：异常不是样本自带的固有属性，而是"相对大多数样本而言"的偏离。同一条数据放在不同的数据集里，可能一处被判异常、另一处被判正常。

### 1.2 离群点长什么样：三种形式

| 形式 | 含义 | 例子 |
|:--- |:--- |:--- |
| **点异常（Point Outlier）** | 单个样本自身反常 | 一笔金额远超历史的交易 |
| **上下文异常（Contextual Outlier）** | 脱离上下文看正常，放进特定上下文才异常 | 夏季气温 10℃ |
| **集合异常（Collective Outlier）** | 单独看都正常，成组出现才反常 | 短时间连续高频登录 |

这三种形式是离群点检测的目标形态，和"训练时有没有标签"无关——本系列主线（无监督）里三种都会遇到。本节末尾配有散点图直观展示它们（见"可视化：三类异常形式"）。

### 1.3 两个必须记住的认知边界

> ⚠️ **高分 ≠ 业务上的真异常。** 模型只在它学到的正常模式范围内判断。高分只说明这个样本和它见过的主体分布差异大；业务上真正的风险事件，也可能因为长得"像正常数据"而拿到低分。模型不知道答案，它只负责排序。

> 💡 **异常 ≠ 脏数据。** 离群点只是偏离常态，需要业务专家复核：一部分是录入错误或测量噪声，一部分才是真实风险事件。

### 1.4 术语边界：本系列说的"离群点检测"不包括什么

这一节放在最后——它不影响上面的核心理解，但读文献和库文档时会反复遇到，所以集中交代清楚：

- **Anomaly Detection（异常检测）是总称**，下面三类任务都在它伞下；
- **监督式异常检测**：训练数据带正常 + 异常标签，本质是**不平衡二分类**；
- **单类 / 新颖性检测**：只用正常样本训练，判断新样本是否"新"，严格叫 **Novelty Detection**；
- **Outlier Detection（离群点检测）**：特指 1.1 描述的**无监督**任务，这也是本系列主线。

> ⚠️ 工程上的宽泛用法：PyOD 全名是 Python Outlier Detection，却把上述方法都收进统一 API；论文里也常把 outlier detection 当整个领域的泛称。本系列正文遵守严格定义——说"离群点检测"只指无监督那一支；需要涵盖全部三类时才说"异常检测"。

## 2. 算法家族全景图：先有地图，再走路线

> 本系列按"一章一个库"组织，是为了贴近工程落地；但如果只按库学习，会有两个副作用：
> ① 同一算法在不同库里重复出现（LOF、IsolationForest 在 PyOD 与 scikit-learn 中都会讲到），读者不清楚"我其实是在学同一个东西"；
> ② 学完能调用 API，却难形成"面对这个问题该用哪类算法"的判断力。
>
> 因此，在讲任何库之前，先给出一棵**算法家族树**。后续每一章都会标注它覆盖了哪些家族。

### 2.1 六大算法家族

| 家族 | 核心思想 | 代表算法 | 擅长场景 | 主要短板 |
|:--- |:--- |:--- |:--- |:--- |
| **统计与概率** | 假设正常数据服从某分布，偏离即异常 | Z-score、HBOS、Mahalanobis 距离、GMM | 低维、分布可假设、需要快速基线 | 高维失效；多峰/复杂分布难建模 |
| **距离与密度** | 异常 = 邻居少或局部密度低 | kNN、LOF、ABOD、CBLOF、COF | 局部异常、簇状数据、可解释性好 | 维度灾难；参数（k）敏感；大规模计算贵 |
| **隔离与集成** | 异常更容易被随机划分隔离出来 | Isolation Forest、Extended IForest | 高维、大数据、速度快、无需缩放 | 对局部密度型异常不敏感 |
| **单类与边界** | 只学正常样本的边界，界外即异常 | One-Class SVM、SVDD | 小样本、只有正常数据、边界清晰 | 核与参数敏感；大规模训练慢 |
| **重构与深度** | 用模型重构正常数据，重构误差大即异常 | AE、VAE、GAN-based、Transformer-based | 图像、高维嵌入、复杂序列 | 需大量正常数据；训练复杂；可解释性弱 |
| **时序与流式** | 利用时间依赖或增量更新捕捉偏离 | STL 残差、ADTK 规则、River 在线模型 | 指标监控、实时告警、概念漂移 | 依赖时间结构假设；评估需防信息泄漏 |

> 💡 这六个家族不是互斥的：Isolation Forest 既是"隔离与集成"，也常被当作"集成化的距离方法"；AE 既属"重构与深度"，也可用作"单类"建模。家族是理解地图，不是严格分类学。

### 2.2 家族与库的对应关系

| 家族 | PyOD | scikit-learn | ADTK | River | Alibi Detect | PyCaret |
|:--- |:---:|:---:|:---:|:---:|:---:|:---:|
| 统计与概率 | ✅ | 部分 | — | ✅ | — | ✅ |
| 距离与密度 | ✅ | ✅ | — | ✅ | — | ✅ |
| 隔离与集成 | ✅ | ✅（IForest） | — | ✅ | — | ✅ |
| 单类与边界 | ✅ | ✅（OCSVM） | — | ✅ | ✅ | ✅ |
| 重构与深度 | 少量 | MLP 相关 | — | 少量 | ✅（核心） | — |
| 时序与流式 | 部分 | — | ✅（核心） | ✅（核心） | ✅ | — |

> 📌 读法：PyOD 与 scikit-learn 覆盖"表格数据"的大部分家族；时序去 ADTK、流式去 River、高维深度去 Alibi Detect；PyCaret 适合快速批量跑基线。第 7 章会给出同一任务跨库的对照实验。

## 3. 选型决策框架：三步定位你的算法

> 面对一个新问题，按下面三步走，而不是从"哪个库最火"开始。

### 3.1 第一步：看数据形态

| 数据形态 | 优先家族 | 首选库 | 说明 |
|:--- |:--- |:--- |:--- |
| 表格 / 低维数值 | 统计与概率、距离与密度、隔离与集成 | PyOD 或 scikit-learn | 起步最快，先跑 IForest + LOF 做基线 |
| 高维稠密（图像、嵌入向量） | 重构与深度、隔离与集成 | Alibi Detect 或 PyOD | 高维下距离失效，优先隔离/重构类 |
| 时间序列（带时间戳） | 时序与流式 | ADTK，或 PyOD + 滑动窗口 | 必须先处理趋势与季节性，避免把周期当异常 |
| 流式逐条到达 | 流式增量 | River | 无法一次性加载全量，需在线更新 |

### 3.2 第二步：看标签情况

| 标签情况 | 可用范式 | 评估方式 |
|:--- |:--- |:--- |
| 完全无标签 | 无监督 | 排序指标 + 合成异常验证 + 业务复核 |
| 只有正常样本 | 半监督 / 单类 | 留出正常样本 + 注入异常构造验证集 |
| 正常 + 异常都有 | 监督，或半监督 | 分层切分 + 分类/排序指标均可用 |

### 3.3 第三步：看实时性与业务约束

| 约束 | 应对方式 |
|:--- |:--- |
| 要求实时告警 | River 在线模型；或离线预计算分数 + 轻量阈值规则 |
| 需要解释告警原因 | 优先可输出特征贡献度的模型；配套 SHAP / LIME |
| 人工复核预算有限 | 用 Precision@k 定阈值，只送 Top-k 给人看 |
| 业务频繁变化 | 引入漂移监控（Evidently / NannyML），定期重训与重定阈值 |
| 标注成本高 | 无监督起步，把人工复核结果回流成标签 |

> 💡 三步走完仍拿不准时：用 PyOD 或 PyCaret 批量跑多个家族的模型，按 PR-AUC / Precision@k 排序，选表现最稳的做基线，再逐步优化。

### 3.4 一个完整决策示例

> 场景：服务器 CPU 指标每分钟一条，历史半年的数据里没有任何异常标签，希望每周人工核查不超过 30 条告警。
>
> - 数据形态 → 时间序列 → 家族选"时序与流式" → 库选 ADTK；
> - 标签情况 → 完全无标签 → 无监督范式 → 用 STL 残差 + 合成异常注入建验证集；
> - 业务约束 → 每周 30 条 → 用 Precision@30 选阈值，而不是用 ROC-AUC 挑模型。

## 4. 异常检测的作用与意义

### 4.1 典型业务场景

- **金融风控**：信用卡欺诈识别、洗钱识别、信贷异常行为
- **工业物联网**：传感器故障、产线质量异常、预测性维护
- **网络安全**：入侵检测、异常访问、爬虫识别
- **医疗检验**：检验指标异常、心电信号异常筛查
- **AIOps 运维**：服务器指标、系统流量、日志时序告警
- **数据预处理**：清洗脏数据，避免异常样本污染回归、聚类、PCA 等下游模型

### 4.2 学术与工程价值

1. 作为数据质量底座，保障后续建模可靠性；
2. 自动捕获稀有高价值事件，降低人工筛查成本；
3. 适配标注稀缺场景，弥补监督学习的短板；
4. 输出连续异常分数（anomaly score），支持判断偏离程度，而不只是 0/1 标签。

## 5. Python 异常检测生态：专用包梳理（按重要性排序）

排序依据：工业落地量、算法丰富度、社区活跃度、文档成熟度。

1. **PyOD — Python Outlier Detection（首选，最重要）**
行业事实标准库，内置近百种离群检测算法，统一 API，支持分数输出，论文对比、工业原型首选。覆盖家族：统计与概率、距离与密度、隔离与集成、单类与边界。
2. **Scikit-learn（内置异常模块）**
不是专门异常包，但自带 IsolationForest、LOF、OneClassSVM，轻量、无需额外重型依赖。适合快速基线与教学演示。
3. **ADTK（Anomaly Detection Toolkit）**
时序异常专项库，面向指标监控场景，内置滑动窗口、时序规则、告警降噪。覆盖家族：时序与流式。
4. **River（原 Creme）**
流式在线学习，适合数据逐条流入、无法一次性加载全量数据的实时检测场景。覆盖家族：流式增量。
5. **Alibi Detect**
深度学习异常检测库：自编码器、VAE、重构误差类模型，适配图像、高维长时序。覆盖家族：重构与深度。
6. **PyCaret Anomaly**
低代码自动化流水线，快速批量对比多个异常模型，适合业务快速验证基线。

> **辅助配套库**（不属于独立异常检测器，但高频搭配）：
> - `imblearn`：不平衡采样
> - `EvidentlyAI` / `NannyML`：数据漂移监控
> - `SHAP` / `LIME`：模型可解释性分析

> 📌 选型速记：只想快速跑通 → scikit-learn；要认真做对比 → PyOD；时序 → ADTK；实时流 → River；高维深度 → Alibi Detect；批量摸底 → PyCaret。

## 6. 评估范式：为什么关注"排序"而非"分类"

### 6.1 离群点检测的本质是排序（Ranking）

> ⚠️ 重要限定：这套排序优先的范式，**只适用于本系列主线——无监督离群点检测**（以及与之同源、同样不产出类别标签的单类新颖性检测）。它们训练时都没有异常标签，模型输出的是分数而非类别。监督式异常检测不属于这个逻辑，它直接按不平衡分类思路评估。

对于无监督 / 单类模型，训练时没有异常标签，模型不知道什么是真异常，它输出的是**连续异常分数 (Anomaly Score)**，只代表样本偏离它学到的正常模式的程度。

模型本身**不存在一个天然、唯一的阈值，直接给出"是/否异常"的判决**。我们核心追求的效果是：**真实异常样本的分数整体高于正常样本**，也就是优秀的排序能力。

最终输出 0/1 告警标签，是工程上人为选定阈值、对分数做切割之后得到的副产品。阈值的选择取决于业务：人工审核成本、漏检代价、误报容忍度。

### 6.2 推荐评估体系

1. **ROC-AUC**：衡量异常分数整体排序能力，值越接近 1 说明异常与正常分离度越好。
2. **PR-AUC（Precision-Recall AUC）**：在极度不平衡场景下，比 ROC 更能反映头部高风险样本的识别能力。
3. **Precision@k**：工业最实用指标。假设只能人工核查前 k 个最高分样本，其中真实异常的占比是多少？

> ⚠️ 注意：以下代码中的 `y_true` 仅用于**离线基准测试**或**半监督验证**。在纯无监督生产环境中，通常没有 `y_true`，需依赖业务反馈闭环或无参考评估方法。

### 6.3 无标签时如何构建验证集（新增重点）

无监督不等于无法评估。常用三种构造方式：

1. **合成异常注入（Synthetic Anomaly Injection）**
在干净的正常样本上施加人为扰动，生成带标签的"伪异常"。例如：对某个特征乘以 3~5 倍、把两个特征对调、用均匀分布替换某个特征区间。
> ⚠️ 局限：合成异常只代表"你想象得到的异常"，对新型异常的高估是常见陷阱。

2. **仅正常样本的交叉验证（Normal-only CV）**
把正常样本切成训练折与验证折，在验证折上计算分数分布，观察是否有少量样本分数显著偏高。适合"数据基本干净、只需揪出极少数坏点"的场景。

3. **时间切分（Time-based Split）**
时序数据必须按时间先后切分，用过去训练、未来验证，**严禁随机打乱**（否则未来信息泄漏，指标虚高）。

4. **业务反馈闭环（延迟标签回流）**
上线后把人工复核结论回流成标签，滚动更新验证集。这是最贴近真实价值的方式，也是长期唯一可靠的评估来源。

### 6.4 阈值选取：分数如何变成告警（新增重点）

> ⚠️ 阈值不是模型自带的属性，是业务权衡的结果。第 7 章会专门展开，这里给出四类常用思路：

| 思路 | 做法 | 适用场景 |
|:--- |:--- |:--- |
| **业务预算驱动** | 每周只能看 k 条 → 取分数 Top-k 作为告警 | 人工成本明确的运营场景 |
| **统计分位数驱动** | 取正常样本分数的 99% / 99.5% 分位为阈值 | 正常样本较干净、分布稳定 |
| **分布拐点驱动** | 在分数分布上找肘部、用 Otsu 或极值理论（EVT / SPOT）自动定阈 | 无业务先验、需要自动校准 |
| **误报率约束驱动** | 给定可容忍误报率上限，反推对应分位数阈值 | 对误报极敏感的场景（如生产停线） |

### 6.5 漂移之后如何持续评估（新增重点）

正常模式会随时间变化，模型上线不等于一劳永逸：

- **监控分数分布偏移**：用 KS 检验、PSI 比较今日与基线期的异常分数分布；
- **监控特征漂移**：用 Evidently / NannyML 跟踪输入特征分布变化；
- **延迟标签回流后重算指标**：用最新复核结论滚动更新 PR-AUC / Precision@k；
- **定期重训与阈值重校准**：漂移达到阈值时触发重训，并重新确定告警阈值。

## 7. 贯穿全系列的六个主题（新增）

> 这些难点不属于任何单一章节，但会在每一章反复出现。先列在这里，后续遇到时回看此表。

| 主题 | 为什么重要 | 主要出现章节 |
|:--- |:--- |:--- |
| **特征缩放** | 距离/密度类算法对尺度敏感，必须用 `RobustScaler`（对异常稳健），而非 `StandardScaler`（均值方差会被异常带偏） | 第 1、2 章 |
| **高维与维度灾难** | 高维下距离趋于相等，LOF / ABOD 等距离密度方法失效 | 第 1、5 章 |
| **类别特征与缺失值** | 真实业务不是纯数值矩阵，需编码与填充策略，且策略本身会影响异常判定 | 第 1、6 章 |
| **`contamination` 参数** | 决定分数到标签的阈值，常被误用为"真实异常比例"；它实际是**预期异常比例的先验假设** | 第 1、2、6 章 |
| **可解释性** | 告警必须能给出理由，否则业务不信任、闭环建不起来 | 第 1、5、7 章 |
| **数据漂移** | 正常模式变化后模型逐渐失效，需要监控与重训机制 | 第 3、4、7 章 |

> 💡 建议：学到任何一章时，都回头对照这张表，确认该章的算法在这六个维度上表现如何。

## 8. 全系列教程章节总规划

> ✅ **统一约定**：每一章聚焦一个包；每一章内部固定结构：
> 包简介 & 适用场景 → 安装环境 → API 思想 → 合成数据集演示 → 核心算法详解 → 可视化 → 评估指标 → 实战案例 → 调参踩坑。
> 每章开头会标注它覆盖了第 2 节算法家族全景图中的哪些家族。

- **第0章（本章）**：导论，概念 + 算法家族地图 + 选型框架 + 生态 + 评估范式 + 路线预告
- **第1章：PyOD** — 全能离群检测工具箱（篇幅最长，核心重点；涵盖半监督微调与模型集成）
- **第2章：Scikit-learn 原生异常模块** — IsolationForest / LOF / OneClassSVM，对比 PyOD 同名实现差异
- **第3章：ADTK** — 时间序列异常检测专项
- **第4章：River** — 流式、在线增量异常检测
- **第5章：Alibi Detect** — 基于深度学习（AE/VAE）的高维异常检测
- **第6章：PyCaret Anomaly** — 低代码快速建模与多模型对比
- **第7章：综合工程实战篇** — 多模型融合、科学阈值选取策略、漂移监控、模型保存与部署
- **附录**：公开数据集清单、评估指标模板、常见坑汇总、**库 × 算法家族 × 场景对照表**

### 系列代码规范约定（后续所有章节统一遵守）

- DataFrame 作为输入主体，贴近真实业务；
- `anomaly_score`：分数越高，样本越异常；
- 标签约定：0 = 正常样本，1 = 异常样本；
- **评估核心原则**：关注异常分数的**排序能力（Ranking）**，优先使用 ROC-AUC、PR-AUC、Precision@k；
- 标准可视化套件：异常散点标记、异常分数直方图、ROC/PR 曲线、特征贡献度条形图。

## 9. 基础环境前置准备

先安装全系列通用基础依赖，后续各包在进入对应章节时再单独安装自身库。

> ⚠️ 建议按顺序执行下面的单元格；`pyod / adtk / river / alibi-detect / pycaret` 等专用包留到对应章节安装，避免一次性装齐导致依赖冲突或环境过重。

In [ ]:
# 基础依赖安装（全系列通用）
!pip install numpy pandas matplotlib scipy scikit-learn

# 专用包按需安装（进入对应章节时再取消注释）
# !pip install pyod          # 第1章
# !pip install adtk          # 第3章
# !pip install river         # 第4章
# !pip install alibi-detect  # 第5章
# !pip install pycaret       # 第6章
# !pip install shap          # 可解释性，第1/5/7章

## 10. 十行快速上手：跑出第一条异常分数（新增）

> 用一个合成二维数据集，跑通 IsolationForest 的完整链路：拟合 → 打分 → 评估。
> 执行下面的单元格，确认输出合理的 ROC-AUC（应接近 1）。

**代码要点**：scikit-learn 的 `decision_function` 返回的分数**越高越正常**，所以取负号转换为本系列约定——**分数越高越异常**。这个符号约定是新手最常踩的坑之一。

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(42)
X = np.vstack([rng.normal(0, 1, (300, 2)),
               rng.uniform(6, 9, (10, 2))])
y = np.array([0] * 300 + [1] * 10)

clf = IsolationForest(n_estimators=100, random_state=42).fit(X)
anomaly_score = -clf.decision_function(X)   # 取负号：分数越高越异常

print("ROC-AUC =", round(roc_auc_score(y, anomaly_score), 3))
print(pd.Series(anomaly_score).describe())

## 11. 可视化：三类异常形式长什么样（新增）

> 一图胜千言。下面用三张散点图分别展示点异常、上下文异常、集合异常。
> 红色 × 为异常点。注意上下文异常在"特征 1"维度上完全正常，只有结合两个特征的关系才能看出来。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
normal = rng.normal(0, 1, (200, 2))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# ① 点异常：单个样本远离主体
ax = axes[0]
ax.scatter(normal[:, 0], normal[:, 1], s=14, color="#8aa8c8", alpha=0.8)
ax.scatter([5.2], [4.8], s=90, color="#d85a30", marker="x", linewidths=2)
ax.set_title("点异常：单个样本远离主体")

# ② 上下文异常：特征1正常，但违背两特征的关系
ax = axes[1]
x = normal[:, 0]
y = 0.8 * x + rng.normal(0, 0.35, 200)
ax.scatter(x, y, s=14, color="#8aa8c8", alpha=0.8)
ax.plot([-3, 3], [-2.4, 2.4], color="#1d9e75", ls="--", lw=1)
ax.scatter([0.1], [4.5], s=90, color="#d85a30", marker="x", linewidths=2)
ax.set_title("上下文异常：x 正常但 y 违背关系")

# ③ 集合异常：单独看都正常，成簇才反常
ax = axes[2]
ax.scatter(normal[:, 0], normal[:, 1], s=14, color="#8aa8c8", alpha=0.8)
cluster = rng.normal(3.0, 0.16, (25, 2))
ax.scatter(cluster[:, 0], cluster[:, 1], s=20, color="#d85a30", alpha=0.85)
ax.set_title("集合异常：成簇出现才反常")

for ax in axes:
    ax.set_xlabel("特征 1")
    ax.set_ylabel("特征 2")
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 12. 公开基准数据集清单（新增）

> 除"合成数据集演示"外，后续实战建议使用公开基准，保证结果可复现、可对比。

| 数据集 | 类型 | 说明 | 获取 |
|:--- |:--- |:--- |:--- |
| **ODDS** | 表格 | 经典离群检测基准集合，含 mnist、yahoo、ionosphere 等 | odds.cs.stonybrook.edu |
| **KDD Cup 1999** | 网络入侵 | 经典大规模入侵检测数据，年代久远但仍是基线常用 | UCI/KDD 归档 |
| **NAB** | 时序 | Numenta Anomaly Benchmark，50+ 真实与合成时序 | GitHub: numenta/NAB |
| **Yahoo Webscope S5** | 时序 | AIOps 指标异常检测基准，含真实标注 | Yahoo Webscope |
| **SMAP / MSL** | 时序 | NASA 航天器遥测异常数据 | NASA 公开数据集 |
| **MVTec AD** | 图像 | 工业缺陷检测基准，适配重构与深度方法 | mvtec.com/datasets

> ⚠️ 使用基准时务必记录随机种子、切分方式与指标口径；不同论文的预处理差异会让数字不可直接比较。

## 13. 下一章预告

下一章：**第1章 PyOD — Python 离群检测全能库**

我们将从合成二维数据集开始，依次上手：Z-score、HBOS、LOF、IsolationForest、ABOD、CBLOF。**对照第 2 节的算法家族图**，你会看到它们分别落在"统计与概率""距离与密度""隔离与集成"三个家族里。

**本章重点突破：**

1. PyOD 统一 API (`fit`, `predict`, `decision_function`) 的深度解析；
2. **数据预处理**：为什么 LOF / KNN 必须配合 `RobustScaler`（回顾第 7 节"贯穿性主题"）；
3. **可解释性初探**：如何通过异常分数反向定位关键特征；
4. **模型集成**：如何使用 PyOD 的 `Combination` 模块提升鲁棒性。

---

> **思考题：**
> 1. 高维数据下 LOF 为什么经常失效？（提示：维度灾难与距离稀疏性，见第 2、7 节）
> 2. 在无标签情况下，如何利用业务常识确定初始告警阈值？（提示：见第 6.4 节四类思路）
> 3. 时序异常和普通表格点异常的核心差异是什么？（提示：上下文依赖性，见第 1.3、3.1 节）
> 4. 为什么异常检测预处理推荐 `RobustScaler` 而不是 `StandardScaler`？（提示：异常会污染均值和方差）
> 5. 如果业务每周只能人工复核 50 条告警，你会用哪个指标选模型、用什么方法定阈值？（提示：Precision@k + 业务预算驱动）